In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from nibabel.processing import resample_to_output

from resources import find_tilt_error

In [2]:
project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)

target_column = "is_pathologic"

labels = labels_df.set_index("uid")[target_column]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))

Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1362
Liczba etykiet: 1362


In [3]:
from resources.standarize_image import standardize_image
from resources.cut_off_edges import cut_off_edges


def preprocess_one_nifti(image):

    image = nib.load(image)

    processed_image = cut_off_edges(image)

    processed_image = cut_off_edges(
        processed_image,
        x_left=35,
        x_right=35,
        y_front=40,
        y_back=40,
        z_down=30,
        z_up=30,
    )
    processed_image = cut_off_edges(
        processed_image,
        x_left=30,
        x_right=30,
        y_front=35,
        y_back=35,
        z_down=25,
        z_up=25,
        z_center_method=find_z_argmax
    )

    processed_image = standardize_image(
        processed_image,
        percent_top=0.02,
    )

    processed_image = resample_to_output(
        processed_image,
        voxel_sizes=(2.46, 2.46, 2.46),
        order=1,
    )

    return processed_image

In [4]:
from resources.rotate_image import rotate_volume
from resources.cut_off_edges import cut_off_edges
from resources.find_z import (
    calculate_z_profile,
    find_z_argmax,
)
from resources.find_y import (
    calculate_y_profile,
    find_y_area_center,
)
from resources.find_x import (
    calculate_x_profile,
    find_x_area_center,
)

import numpy as np


def calculate_similarity(volume, template_volume):
    """
    Suma iloczynów odpowiadających sobie wokseli.
    Większy wynik oznacza lepsze nałożenie aktywnych obszarów.
    """
    volume_flat = np.asarray(
        volume,
        dtype=np.float32,
    ).ravel()

    template_flat = np.asarray(
        template_volume,
        dtype=np.float32,
    ).ravel()

    if volume_flat.shape != template_flat.shape:
        raise ValueError(
            f"Różne rozmiary wektorów: "
            f"volume={volume_flat.shape}, "
            f"template={template_flat.shape}"
        )

    return float(
        np.sum(volume_flat * template_flat)
    )


def find_pitch_error_by_similarity(
    image,
    template_image,
):
    # Obraz przed końcowym przycięciem.
    volume = image.get_fdata(
        dtype=np.float32,
    )

    # Środek obrotu wyznaczamy na obrazie nieprzyciętym.
    x_profile = calculate_x_profile(volume)
    x_centre = find_x_area_center(x_profile)

    y_profile = calculate_y_profile(volume)
    y_centre = find_y_area_center(y_profile)

    z_profile = calculate_z_profile(volume)
    z_centre = find_z_argmax(z_profile)

    centre = (
        x_centre,
        y_centre,
        z_centre,
    )

    # Template nie jest obracany, więc przycinamy go tylko raz.
    cropped_template_image = cut_off_edges(
        template_image,
        x_left=25,
        x_right=25,
        y_front=30,
        y_back=30,
        z_down=20,
        z_up=20,
        z_center_method=find_z_argmax
    )

    template_volume = (
        cropped_template_image.get_fdata(
            dtype=np.float32,
        )
    )

    best_similarity = -np.inf
    best_angle = 0

    for angle in range(-25, 26):
        # Najpierw obracamy pełny obraz.
        rotated_image = rotate_volume(
            image=image,
            angle=angle,
            axis="x",
            center=centre,
        )

        # Dopiero później usuwamy czarne krawędzie.
        cropped_rotated_image = cut_off_edges(
            rotated_image,
            x_left=25,
            x_right=25,
            y_front=30,
            y_back=30,
            z_down=20,
            z_up=20,
            z_center_method=find_z_argmax
        )

        rotated_volume = (
            cropped_rotated_image.get_fdata(
                dtype=np.float32,
            )
        )

        if rotated_volume.shape != template_volume.shape:
            raise ValueError(
                f"Różne rozmiary po przycięciu: "
                f"image={rotated_volume.shape}, "
                f"template={template_volume.shape}"
            )

        similarity = calculate_similarity(
            rotated_volume,
            template_volume,
        )

        if similarity > best_similarity:
            best_similarity = similarity
            best_angle = angle

    return best_angle

In [5]:
template_path_1 = (
    "C:/Users/abram/PycharmProjects/"
    "DrivenData_DAT_Parkinsons/src/resources/"
    "template_94xfri09.nii.gz"
)
template_path_2 = (
    "C:/Users/abram/PycharmProjects/"
    "DrivenData_DAT_Parkinsons/src/resources/"
    "template_09lzrk79.nii.gz"
)

template_path_3 = (
    "C:/Users/abram/PycharmProjects/"
    "DrivenData_DAT_Parkinsons/src/resources/"
    "template_2c9ros7y.nii.gz"
)
template_path_4 = (
    "C:/Users/abram/PycharmProjects/"
    "DrivenData_DAT_Parkinsons/src/resources/"
    "template_0ezzf8s2.nii.gz"
)
template_path_5 = (
    "C:/Users/abram/PycharmProjects/"
    "DrivenData_DAT_Parkinsons/src/resources/"
    "template_33mg1zaq.nii.gz"
)

template_paths = [
    template_path_1,
    template_path_2,
    template_path_3,
    template_path_4,
    template_path_5,
]

template_images = [
    preprocess_one_nifti(template_path)
    for template_path in template_paths
]

In [ ]:
from pathlib import Path

images_dir = Path(
    "C:/Users/abram/PycharmProjects/"
    "DrivenData_DAT_Parkinsons/data/niftis"
)

files = sorted(
    images_dir.glob("*.nii.gz")
)

print("Liczba obrazów:", len(files))

pitch_errors = {}

for index, file_path in enumerate(files, start=1):
    uid = file_path.name.replace(".nii.gz", "")

    print(
        f"{index}/{len(files)} | {uid}",
        flush=True,
    )

    processed_image = preprocess_one_nifti(
    file_path
    )

    pitch_errors = []

    for template_image in template_images:
        pitch_error = (
            find_pitch_error_by_similarity(
                processed_image,
                template_image,
            )
        )

        pitch_errors.append(
            pitch_error
        )

    median_pitch_error = int(
        np.median(pitch_errors)
    )

    print(
        f"pitch_errors = {pitch_errors}",
        flush=True,
    )

    print(
        f"median_pitch_error = "
        f"{median_pitch_error}",
        flush=True,
    )

Liczba obrazów: 1362
1/1362 | 01nouhtc
pitch_errors = [7, -10, -11, -10, -8]
median_pitch_error = -10
2/1362 | 0224wk0y
pitch_errors = [0, 0, 0, 2, 0]
median_pitch_error = 0
3/1362 | 049enulq
pitch_errors = [11, 5, 4, 6, 7]
median_pitch_error = 6
4/1362 | 04x0qzfh
pitch_errors = [9, 9, 7, 7, 3]
median_pitch_error = 7
5/1362 | 05xt54fn
pitch_errors = [9, 0, 1, 1, 1]
median_pitch_error = 1
6/1362 | 06akt7zq
pitch_errors = [6, 0, -1, 0, 6]
median_pitch_error = 0
7/1362 | 06nkwfg3
pitch_errors = [9, 8, 1, 6, 7]
median_pitch_error = 7
8/1362 | 07gbasf5
pitch_errors = [20, 17, 17, 18, 19]
median_pitch_error = 18
9/1362 | 07s387j5
pitch_errors = [2, 2, 2, 2, 2]
median_pitch_error = 2
10/1362 | 085rntdr
pitch_errors = [-5, -10, -12, -12, -12]
median_pitch_error = -12
11/1362 | 08liuqdn
pitch_errors = [6, -6, -9, -4, -6]
median_pitch_error = -6
12/1362 | 08uhbxlb
pitch_errors = [12, 12, 1, 13, 13]
median_pitch_error = 12
13/1362 | 09lzrk79
pitch_errors = [12, 0, -3, 6, 6]
median_pitch_error = 6